# The Shape of Discovery — Colab Environment Setup

Run this notebook first in every Colab session. It clones the private repo, installs dependencies, and restores any previously saved results from Google Drive.

**First time only:** You need to add your GitHub token as a Colab secret.
1. Click the 🔑 key icon in the left sidebar
2. Add a secret named `GITHUB_TOKEN`
3. Paste your fine-grained personal access token (from emergent-inquiry account)
4. Toggle "Notebook access" ON

## Step 1: Mount Google Drive
Results persist here between sessions.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Create project directory on Drive if it doesn't exist
!mkdir -p /content/drive/MyDrive/shape-of-discovery/data
!mkdir -p /content/drive/MyDrive/shape-of-discovery/figures
print('Drive mounted and directories ready.')

## Step 2: Clone the private repo

In [ ]:
import os
from google.colab import userdata

token = userdata.get('GITHUB_TOKEN')

# Clone or pull latest
if not os.path.exists('/content/the-shape-of-discovery'):
    !git clone https://{token}@github.com/emergent-inquiry/the-shape-of-discovery.git /content/the-shape-of-discovery
    print('Repo cloned.')
else:
    %cd /content/the-shape-of-discovery
    !git pull
    print('Repo updated.')

%cd /content/the-shape-of-discovery

## Step 3: Install dependencies

In [ ]:
!pip install -q -r requirements.txt
print('Dependencies installed.')

## Step 4: Restore previous results from Drive
If you've run notebooks before, this copies saved data and figures back into the working directory.

In [ ]:
import shutil
import glob

drive_data = '/content/drive/MyDrive/shape-of-discovery/data'
drive_figs = '/content/drive/MyDrive/shape-of-discovery/figures'
local_data = '/content/the-shape-of-discovery/data'
local_figs = '/content/the-shape-of-discovery/figures'

# Restore data files
data_files = glob.glob(f'{drive_data}/*')
for f in data_files:
    dest = os.path.join(local_data, os.path.basename(f))
    if not os.path.exists(dest):
        if os.path.isdir(f):
            shutil.copytree(f, dest)
        else:
            shutil.copy2(f, dest)
        print(f'  Restored: {os.path.basename(f)}')

# Restore figures
fig_files = glob.glob(f'{drive_figs}/*')
for f in fig_files:
    dest = os.path.join(local_figs, os.path.basename(f))
    if not os.path.exists(dest):
        shutil.copy2(f, dest)
        print(f'  Restored: {os.path.basename(f)}')

if not data_files and not fig_files:
    print('No previous results to restore. Fresh start.')
else:
    print(f'Restored {len(data_files)} data items and {len(fig_files)} figures.')

## Step 5: Verify environment

In [ ]:
import psutil
import platform

ram_gb = psutil.virtual_memory().total / (1024**3)
cpu_count = psutil.cpu_count()

print(f'Python:    {platform.python_version()}')
print(f'RAM:       {ram_gb:.1f} GB')
print(f'CPUs:      {cpu_count}')
print(f'Working:   {os.getcwd()}')
print()

# Check key packages
packages = ['pandas', 'numpy', 'scipy', 'networkx', 'ripser', 'persim', 'sklearn', 'matplotlib']
for pkg in packages:
    try:
        mod = __import__(pkg)
        ver = getattr(mod, '__version__', '?')
        print(f'  {pkg}: {ver}')
    except ImportError:
        print(f'  {pkg}: NOT INSTALLED')

print()

# Check data status
data_files = os.listdir('data')
parquets = [f for f in data_files if f.endswith('.parquet')]
if parquets:
    print(f'Data ready: {len(parquets)} parquet files found')
    for f in sorted(parquets):
        size_mb = os.path.getsize(f'data/{f}') / (1024**2)
        print(f'  {f}: {size_mb:.1f} MB')
else:
    print('No data yet. Run 00_data_acquisition.py first.')

print()
print('Environment ready. You can now run notebooks.')

## Save results back to Drive
Run this cell **after** completing a notebook to persist results.

In [ ]:
import shutil
import glob
import os

drive_data = '/content/drive/MyDrive/shape-of-discovery/data'
drive_figs = '/content/drive/MyDrive/shape-of-discovery/figures'

# Save data
saved = 0
for f in glob.glob('data/*'):
    if f.endswith('.gitkeep'):
        continue
    dest = os.path.join(drive_data, os.path.basename(f))
    if os.path.isdir(f):
        if os.path.exists(dest):
            shutil.rmtree(dest)
        shutil.copytree(f, dest)
    else:
        shutil.copy2(f, dest)
    saved += 1

# Save figures
for f in glob.glob('figures/*'):
    if f.endswith('.gitkeep'):
        continue
    shutil.copy2(f, os.path.join(drive_figs, os.path.basename(f)))
    saved += 1

print(f'Saved {saved} files to Google Drive.')
print('Your results will survive session disconnects.')

---

## Workflow Reminder

1. **Start of session:** Run cells 1-5 above (mount → clone → install → restore → verify)
2. **Run a notebook:** Open the notebook you want to run (NB00, NB01, etc.)
3. **End of session:** Come back here and run the "Save results" cell
4. **Next session:** Steps 1-5 again — your data restores automatically

### Running notebooks in Colab

Option A — Open directly:
```
File → Open notebook → GitHub tab → paste repo URL → select notebook
```

Option B — Run from this notebook:
```python
# For the data acquisition script:
!python 00_data_acquisition.py

# For Jupyter notebooks:
!jupyter nbconvert --execute --to notebook --inplace 01_patent_atlas.ipynb
```

Option C — Convert to Colab-native:
Upload the .ipynb files directly to Colab and run interactively.